# Module 3.1: Build a Grounded Booking Agent with GraphRAG

**Purpose:** Turn the GraphRAG search patterns from Module 2 into a grounded booking workflow. The agent answers hotel questions from Neo4j data, and a protected database command saves booking requests safely.

Module 2 showed several ways to find hotel information in the graph. This notebook uses two of those read paths in one agent. The agent chooses the read path that fits the question, uses the returned Neo4j data to answer, and says when the graph does not have the needed fact.

Use your configured **Neo4j database** and **Amazon Bedrock**. This notebook does not create AWS resources.

**Brief overview**

- **GraphRAG:** A way to answer a question with information retrieved from a graph and its source documents.
- **Grounded booking workflow:** An agent answers hotel questions from retrieved Neo4j data, and a protected database command saves booking requests safely.
- **Tool choice:** The model picks the best of two ways to read the graph.
- **Hybrid search:** A search that combines meaning-based matching with exact-word matching.
- **Safe retry:** Sending the same booking request again returns the first result instead of creating a second booking.
- **Guest limit:** A Neo4j rule that rejects a booking request for more than 10 guests.

## What Neo4j and AWS do

| Neo4j | AWS |
|---|---|
| Stores hotel facts, such as amenities, ratings, and policies | Amazon Bedrock reads the retrieved facts and writes an answer |
| Finds matching hotel text with vector and full-text indexes | Amazon Nova 2 turns the question into a search vector |
| Runs the reviewed graph query that connects a source passage to its hotel |  |
| Stores the guest-limit rule and saves booking requests safely |  |

Module 2 explains the GraphRAG read paths. This notebook turns them into the two tools the booking agent uses.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("03-grounded-booking-agent")
print(f"Workshop root: {REPO_ROOT}")


In [ ]:
import inspect
import json
import os
import uuid
from datetime import date, timedelta

import boto3

from workshop.agent_tools import PASSAGE_TOOL, READ_TOOLS, RECORD_TOOL
from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id
from workshop.contracts import (
    MAX_GUESTS,
    OVER_LIMIT_GUESTS,
    ReservationReason,
    ReservationStatus,
)
from workshop.fixtures import (
    HERO_NAME,
    HERO_SOURCE,
    apply_reservation_fixtures,
    load_manifest,
    readiness_problems,
)
from workshop.grounding import MISSING_LIVE_AVAILABILITY
from workshop.hybrid_retrieval import Neo4jConfig, search_hotel_knowledge
from workshop.prompts import BASE_GROUNDING_PROMPT
from workshop.retrieval_setup import report_problems
from workshop.workshop_utils import (
    ToolTraceHook,
    selected_tool_names,
    show_result,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = configure_aws_region()
MODEL_ID = default_model_id()
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured, so live cells will be skipped. Set NEO4J_URI/USERNAME/PASSWORD/DATABASE.")
if not BEDROCK_READY:
    print("AWS credentials are not configured, so live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to retrieve the fixture hotel.")

## 1. Prepare the graph for hotel search and booking

**Purpose:** Add the small amount of setup data that the booking examples need.

Run the cell below. You can run it more than once.

- **Example hotel IDs:** Stable internal IDs for the hotels used in this notebook. The booking command uses an ID so it saves the request for the correct hotel.
- **Uniqueness checks:** Database rules that prevent two bookings from using the same request ID.
- **Guest-limit rule:** A database rule that sets the maximum number of guests for one booking request. This workshop sets the limit to 10.
- **Readiness check:** A check that both search indexes work and that the example hotel exists.

This setup does not change the hotel names, amenities, ratings, or other source facts.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_reservation_fixtures(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. Search for one hotel's amenities and rating

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

**Purpose:** Find facts about one hotel and show the source that supports them.

Run this search. Exact-word search finds the hotel name. Meaning-based search finds text about amenities and ratings. The graph then returns the hotel, its amenities, its rating, and its stable `hotel_id`. The `hotel_id` is the internal ID used for booking requests.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hotel details question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']} | hotel_id={top['hotel_id']}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Amenities: {', '.join(top['amenities'])}")
    print("\nSource chunk text (the provenance for these facts):")
    print(top["chunk_text"][:600])

### How the search finds hotel facts

**Purpose:** Show how the search connects your question to the saved hotel facts.

- **Meaning-based search:** Finds text with a similar meaning to your question.
- **Exact-word search:** Finds the hotel name and other exact words.
- **Graph lookup:** Follows a matching source passage to its hotel record. It returns the amenities, rating, and `hotel_id`.
- **Source passage:** The original text that supports the returned hotel facts.

The notebook uses the same search settings every time. This keeps the returned data format consistent. The model may phrase its final answer differently on different runs.

You have now run one GraphRAG search yourself. Next, you will give the agent two GraphRAG tools and let it choose the right one.

## 3. Give the agent two ways to read the graph

**Purpose:** Let the model choose the right way to find the answer.

- **`search_hotel_passages`:** Finds source text and related facts for a named hotel or a policy.
- **`query_hotel_records`:** Counts, filters, ranks, or averages saved hotel records.

### What happens when you ask a question

1. You ask a question.
2. The model reads each tool's name, description, and input.
3. The model chooses a tool and sends it the question.
4. Strands runs the tool and returns the result to the model.
5. The model writes an answer using the result.

The agent uses a tool when it needs a hotel fact. It replies directly when no hotel fact is needed.

### Strands agent basics

**Brief overview**

- **`Agent`:** The part that sends your question to the model and runs a selected tool.
- **`BedrockModel`:** The connection from the agent to an Amazon Bedrock model.
- **`@tool`:** Python code that the model can choose to run.
- **`ToolTraceHook`:** A helper that shows each tool call and its result.

In [ ]:
print(f"The model receives {len(READ_TOOLS)} tool specifications.\n")
for read_tool in READ_TOOLS:
    print(json.dumps(read_tool.tool_spec, indent=2))
    print()

print("The wrapper behind the first specification:\n")
print(inspect.getsource(READ_TOOLS[0]))

### What the model reads

**Purpose:** Show the information the model uses to choose a tool.

The model receives each tool's name, description, and input fields. The printed tool specification shows the exact information it receives.

The tools reject a blank question. Both tools return the same basic result:

- **`ok`:** Whether the tool completed its work.
- **Evidence:** The passages or graph records the tool found.
- **`grounding_result`:** Whether the graph has enough information to answer the question.
- **`missing_fact`:** The fact the graph needs but does not have.

### Why one question can call a model twice

**Purpose:** Explain the extra step used for questions that need counts, filters, or averages.

`query_hotel_records` uses a model to write a Cypher query.

1. The agent model chooses `query_hotel_records`.
2. The tool asks a model to write one Cypher read query.
3. Neo4j checks the query plan. It runs the query only when it can read data without changing data.
4. The tool returns the Cypher query and its results.

Read the returned Cypher before trusting the answer. A query can run successfully and still ask the wrong question.

### Empty results and errors

- **Empty `records`:** The query ran but found no matching records. Check the Cypher before deciding that the graph has no answer.
- **Error result:** The tool could not create or run a safe read query. It returns `ok: false` and an error code.

In [ ]:
from strands import Agent
from strands.models import BedrockModel


def build_agent():
    """Build one fresh agent and its own trace.

    A fresh agent per question is what makes the routing examples honest. An
    agent that already answered a passage question carries that exchange in
    its messages, and the next choice is then partly an echo of the last one.
    The trace is per agent for the same reason: recorded calls from an earlier
    question have no business in a later check.
    """
    trace = ToolTraceHook()
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=list(READ_TOOLS),
        system_prompt=BASE_GROUNDING_PROMPT,
        hooks=[trace],
    )
    return agent, trace


def recorded_payloads(trace):
    """Return the complete JSON payload of every tool call a trace recorded."""
    return [
        payload
        for call in trace.calls
        for payload in call["payloads"]
        if isinstance(payload, dict)
    ]


print("System prompt sent with every question:\n")
print(BASE_GROUNDING_PROMPT)

## 4. Check which tool the agent chooses

**Purpose:** Check whether the model picks the tool that fits each question.

Each question starts with a new agent. Read the line below each question to see the selected tool and its result.

| Question | Expected tool |
|---|---|
| Amenities and guest rating for one named hotel | `search_hotel_passages` |
| Average guest rating of the hotels in Paris | `query_hotel_records` |
| Count of hotels that offer a spa | `query_hotel_records` |
| The recorded wording of a cancellation policy | `search_hotel_passages` |

The Paris average and spa-count questions are examples in the tool's instructions. Try another count, filter, or average question to test the tool choice with a new question.

The model can choose a different tool on another run. If it chooses the wrong tool, improve the tool name or description.

In [ ]:
ROUTING_CASES = (
    (HERO_QUESTION, PASSAGE_TOOL),
    ("What is the average guest rating of the hotels in Paris?", RECORD_TOOL),
    ("How many hotels offer a spa?", RECORD_TOOL),
    (
        f"What is the cancellation policy at {HERO_NAME}? "
        "Quote the recorded wording.",
        PASSAGE_TOOL,
    ),
)

if not RETRIEVAL_READY:
    print("Skipping the routing table: retrieval is not configured.")
else:
    matched = 0
    for question, expected in ROUTING_CASES:
        agent, trace = build_agent()
        print(f"\nQ: {question}")
        routing_result = agent(question)
        chosen = selected_tool_names(routing_result)
        show_result(routing_result)
        if expected in chosen:
            matched += 1
            print(f"   ✅ expected {expected}, used {chosen}")
        else:
            print(f"   ⚠️  expected {expected}, used {chosen or 'no tool'}")

    print(f"\n{matched} of {len(ROUTING_CASES)} questions reached the expected tool.")
    print("A mismatch is a tool description to improve, not a broken notebook.")

## 5. Ask a question the graph cannot answer

**Purpose:** Show how the agent responds when Neo4j does not contain a needed fact.

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph stores hotel details and total room capacity. It does not store live room availability. The agent can find facts about the hotel, but it cannot confirm whether a room is open next weekend.

The tool result says this clearly:

- **`answerable: false`:** The graph does not have enough information to answer the question.
- **`missing_fact: live_room_availability`:** The missing information is current room availability.

The check below reads this result. It checks the tool's decision instead of the model's exact wording.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping the availability question: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {AVAILABILITY_QUESTION}")
    availability_result = agent(AVAILABILITY_QUESTION)
    show_result(availability_result)

    verdicts = [
        payload["grounding_result"]
        for payload in recorded_payloads(trace)
        if isinstance(payload.get("grounding_result"), dict)
    ]
    print(f"\nTools used: {selected_tool_names(availability_result) or 'none'}")
    print(f"Verdicts returned: {json.dumps(verdicts)}")

    problems = []
    if not trace.calls:
        problems.append("the agent answered an availability question with no tool call")
    if not any(
        verdict.get("missing_fact") == MISSING_LIVE_AVAILABILITY
        for verdict in verdicts
    ):
        problems.append(
            f"no tool result reported missing_fact={MISSING_LIVE_AVAILABILITY}"
        )
    report_problems(problems, "a tool ran and reported live room availability as unsupported.")

## 6. Answer a message that needs no hotel facts

**Purpose:** Confirm that the agent does not search the graph for a simple social message.

> **thanks, that is all**

The agent should reply directly. It does not need to call a tool because the message asks for no hotel facts.

In [ ]:
SOCIAL_TURN = "thanks, that is all"

if not RETRIEVAL_READY:
    print("Skipping the social turn: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {SOCIAL_TURN}")
    social_result = agent(SOCIAL_TURN)
    show_result(social_result)

    used = selected_tool_names(social_result)
    if used:
        print(f"   ⚠️  used {used}; a thank-you needs no hotel fact")
    else:
        print("   ✅ no tool call: the model answered without reading the graph.")

## 7. Reject a booking request that exceeds the guest limit

**Purpose:** Show that Neo4j blocks a booking request for too many guests.

The guest-limit rule allows up to 10 guests. This example asks for 15 guests. Neo4j checks the rule while saving the request, then rejects the request before it creates a booking record.

This section needs a Neo4j connection. It does not use Bedrock. The notebook uses a known example hotel ID and future dates.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id from fixture manifest: {hero_id}")
    print(f"Caller-created request_id for retries: {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

    assert rejected["status"] == ReservationStatus.REJECTED.value, rejected
    assert rejected["reason_code"] == ReservationReason.MAX_GUESTS_EXCEEDED.value, rejected
    assert rejected["hotel_id"] == hero_id, rejected
    assert rejected["max_guests"] == MAX_GUESTS, rejected

## 8. Create one booking request and safely send it again

**Purpose:** Show that a repeated request does not create a duplicate booking.

Send a request for 10 guests, then send the same request again with the same `request_id`.

- **First request:** Creates one booking request for the hotel.
- **Repeated request:** Returns the original request with `duplicate=true`.
- **`request_id`:** A caller-created ID that identifies one booking request. Neo4j allows only one saved request with that ID.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    assert accepted["status"] == ReservationStatus.ACCEPTED.value, accepted
    assert accepted["hotel_id"] == hero_id, accepted
    assert accepted["duplicate"] is False, accepted

    assert replay["status"] == ReservationStatus.ACCEPTED.value, replay
    assert replay["hotel_id"] == hero_id, replay
    assert replay["duplicate"] is True, replay

## 9. Verify the booking request in Neo4j

**Purpose:** Confirm that Neo4j saved one accepted request for one hotel.

Use the `request_id` from the last step to find the saved request.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## Continue to the next modules

**Purpose:** Build on this local agent.

- **Module 4:** Moves the two read tools to AWS Lambda and AgentCore Gateway.
- **Module 5:** Packages the agent for AgentCore Runtime.
- **Module 6:** Adds memory for each user across sessions, with sources and corrections.

This notebook runs against your own Aura database and Amazon Bedrock. It does not create AWS resources.